# 4.3.1. Простые Ising-модели с известным оптимумом

Цель этой тетрадки - проверить корректность реализации bSB и dSB на малых Ising-моделях, для которых точный минимум можно получить аналитически или полным перебором.

На этом этапе рассматривается только чистая Ising-задача:

$$
E(s) = - \sum_{i<j} J_{ij}s_i s_j - \sum_i h_i s_i,
\qquad s_i \in \{-1, +1\}.
$$

QUBO-представление, dummy node, market graph и tabu list здесь не используются.

In [20]:
import numpy as np
import pandas as pd
import itertools
import time

import sys

sys.path.append('курсовая_2')

from SB_modules import BSB, DSB

In [21]:
def ising_energy(s, J, h=None):
    """
    Считает Ising-энергию:

        E(s) = - sum_{i<j} J_ij s_i s_j - sum_i h_i s_i

    Parameters
    ----------
    s : array-like, shape (N,)
        Вектор спинов {-1, +1}.
    J : array-like, shape (N, N)
        Матрица взаимодействий.
    h : array-like, shape (N,), optional
        Вектор локальных полей.

    Returns
    -------
    float
        Значение энергии.
    """
    s = np.asarray(s, dtype=int).reshape(-1)
    J = np.asarray(J, dtype=float)

    if h is None:
        h = np.zeros_like(s, dtype=float)
    else:
        h = np.asarray(h, dtype=float).reshape(-1)

    pair_term = 0.0
    N = len(s)

    for i in range(N):
        for j in range(i + 1, N):
            pair_term += J[i, j] * s[i] * s[j]

    field_term = np.dot(h, s)

    return float(-pair_term - field_term)

In [22]:
def brute_force_ising(J, h=None):
    """
    Полный перебор всех 2^N конфигураций Ising-модели.

    Возвращает:
        best_energy : точный минимум энергии
        best_states : список всех конфигураций, достигающих минимума
    """
    J = np.asarray(J, dtype=float)
    N = J.shape[0]

    if h is None:
        h = np.zeros(N, dtype=float)
    else:
        h = np.asarray(h, dtype=float).reshape(-1)

    best_energy = np.inf
    best_states = []

    for s_tuple in itertools.product([-1, 1], repeat=N):
        s = np.array(s_tuple, dtype=int)
        E = ising_energy(s, J, h)

        if E < best_energy - 1e-12:
            best_energy = E
            best_states = [s]
        elif abs(E - best_energy) <= 1e-12:
            best_states.append(s)

    return best_energy, best_states

In [23]:
def run_sb_many(
    J,
    h=None,
    variant="BSB",
    n_runs=50,
    n_iter=200,
    dt=0.65,
    seed=42,
):
    """
    Запускает BSB или DSB несколько раз с разными seed.

    Возвращает DataFrame с результатами всех запусков.
    """
    J = np.asarray(J, dtype=float)
    N = J.shape[0]

    if h is None:
        h = np.zeros((N, 1), dtype=float)
    else:
        h = np.asarray(h, dtype=float).reshape(N, 1)

    rows = []

    for r in range(n_runs):
        run_seed = seed + r

        if variant.upper() == "BSB":
            solver = BSB(J, h=h, n_iter=n_iter, dt=dt, seed=run_seed)
        elif variant.upper() == "DSB":
            solver = DSB(J, h=h, n_iter=n_iter, dt=dt, seed=run_seed)
        else:
            raise ValueError("variant must be 'BSB' or 'DSB'")

        start = time.perf_counter()
        solver.run(record_trajectory=False)
        elapsed = time.perf_counter() - start

        x_final = solver.x.reshape(-1)
        s_final = np.where(x_final >= 0, 1, -1)

        E = ising_energy(s_final, J, h.reshape(-1))

        rows.append({
            "variant": variant.upper(),
            "run": r,
            "seed": run_seed,
            "energy": E,
            "state": tuple(s_final.tolist()),
            "runtime_sec": elapsed,
        })

    return pd.DataFrame(rows)

In [24]:
def evaluate_instance(
    name,
    J,
    h=None,
    n_runs=50,
    n_iter=200,
    dt=0.65,
    seed=42,
):
    """
    Для одной Ising-задачи:
    1. находит точный оптимум brute force;
    2. запускает BSB;
    3. запускает DSB;
    4. возвращает краткую таблицу.
    """
    E_exact, exact_states = brute_force_ising(J, h)

    all_results = []

    for variant in ["BSB", "DSB"]:
        df = run_sb_many(
            J=J,
            h=h,
            variant=variant,
            n_runs=n_runs,
            n_iter=n_iter,
            dt=dt,
            seed=seed,
        )

        df["gap"] = df["energy"] - E_exact
        success_rate = np.mean(np.isclose(df["energy"], E_exact))

        all_results.append({
            "instance": name,
            "N": np.asarray(J).shape[0],
            "variant": variant,
            "E_exact": E_exact,
            "E_best": df["energy"].min(),
            "mean_energy": df["energy"].mean(),
            "mean_gap": df["gap"].mean(),
            "success_rate": success_rate,
            "mean_runtime_sec": df["runtime_sec"].mean(),
            "n_exact_states": len(exact_states),
        })

    return pd.DataFrame(all_results)

# Тест 1: две ферромагнитные спины

Ферромагнитная связь означает, что минимум достигается при одинаковых спинах:

$$
s_1 = s_2.
$$

При $J_{12}$ = 1:
$$
E(s)=-J_{12} s_1 s_2.
$$

Минимум равен -1, состояния минимума: (1,1) и (-1,-1).

In [27]:
J_ferro_2 = np.array([
    [0.0, 1.0],
    [1.0, 0.0],
])

h_ferro_2 = np.zeros(2)

E_exact, states_exact = brute_force_ising(J_ferro_2, h_ferro_2)

E_exact, states_exact

(-1.0, [array([-1, -1]), array([1, 1])])

## Запуск bSB, dSB

In [28]:
res_ferro_2 = evaluate_instance(
    name="2-spin ferromagnetic",
    J=J_ferro_2,
    h=h_ferro_2,
    n_runs=50,
    n_iter=200,
    dt=0.65,
    seed=42,
)

res_ferro_2

,instance,N,variant,E_exact,E_best,mean_energy,mean_gap,success_rate,mean_runtime_sec,n_exact_states
0,2-spin ferromagnetic,2,BSB,-1.0,-1.0,-1.0,0.0,1.0,0.006221,2
1,2-spin ferromagnetic,2,DSB,-1.0,-1.0,-0.6,0.4,0.8,0.005938,2


In [29]:
df_dsb_ferro = run_sb_many(
    J=J_ferro_2,
    h=h_ferro_2,
    variant="DSB",
    n_runs=50,
    n_iter=200,
    dt=0.65,
    seed=42,
)

df_dsb_ferro["state"].value_counts()

state
(1, 1)      21
(-1, -1)    19
(-1, 1)      5
(1, -1)      5
Name: count, dtype: int64

In [30]:
def sweep_dsb_params_for_instance(
    J,
    h=None,
    dt_values=(0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 0.65, 0.8, 1.0),
    n_iter_values=(50, 100, 200, 500, 1000),
    n_runs=50,
    seed=42,
):
    E_exact, _ = brute_force_ising(J, h)

    rows = []

    for dt in dt_values:
        for n_iter in n_iter_values:
            df = run_sb_many(
                J=J,
                h=h,
                variant="DSB",
                n_runs=n_runs,
                n_iter=n_iter,
                dt=dt,
                seed=seed,
            )

            success_rate = np.mean(np.isclose(df["energy"], E_exact))
            mean_gap = np.mean(df["energy"] - E_exact)

            rows.append({
                "dt": dt,
                "n_iter": n_iter,
                "success_rate": success_rate,
                "mean_gap": mean_gap,
                "E_best": df["energy"].min(),
                "mean_energy": df["energy"].mean(),
            })

    return pd.DataFrame(rows)

In [31]:
dsb_sweep_ferro_2 = sweep_dsb_params_for_instance(
    J=J_ferro_2,
    h=h_ferro_2,
    n_runs=50,
    seed=42,
)

dsb_sweep_ferro_2.sort_values(
    ["success_rate", "mean_gap"],
    ascending=[False, True]
).head(15)

,dt,n_iter,success_rate,mean_gap,E_best,mean_energy
1,0.02,100,1.0,0.0,-1.0,-1.0
2,0.02,200,1.0,0.0,-1.0,-1.0
3,0.02,500,1.0,0.0,-1.0,-1.0
4,0.02,1000,1.0,0.0,-1.0,-1.0
5,0.05,50,1.0,0.0,-1.0,-1.0
6,0.05,100,1.0,0.0,-1.0,-1.0
7,0.05,200,1.0,0.0,-1.0,-1.0
8,0.05,500,1.0,0.0,-1.0,-1.0
9,0.05,1000,1.0,0.0,-1.0,-1.0
14,0.10,1000,1.0,0.0,-1.0,-1.0


In [32]:
dsb_sweep_ferro_2.sort_values(
    ["success_rate", "mean_gap"],
    ascending=[False, True]
).tail(15)

,dt,n_iter,success_rate,mean_gap,E_best,mean_energy
15,0.20,50,0.92,0.16,-1.0,-0.84
25,0.50,50,0.86,0.28,-1.0,-0.72
30,0.65,50,0.82,0.36,-1.0,-0.64
35,0.80,50,0.82,0.36,-1.0,-0.64
32,0.65,200,0.80,0.40,-1.0,-0.60
31,0.65,100,0.70,0.60,-1.0,-0.40
44,1.00,1000,0.68,0.64,-1.0,-0.36
43,1.00,500,0.66,0.68,-1.0,-0.32
36,0.80,100,0.60,0.80,-1.0,-0.20
38,0.80,500,0.60,0.80,-1.0,-0.20


In [33]:
dt = 0.05
n_iter = 1000

Параметры dt=0.65, n_iter=200 подходят для bSB на простом тесте, но не являются устойчивыми для dSB; для первичной верификации dSB требуется меньший шаг интегрирования или большее число итераций.

## Тест 1 после выбора норм параметров для DSB на 2-спиновом ферромагнетике

In [34]:
res_ferro_2 = evaluate_instance(
    name="2-spin ferromagnetic",
    J=J_ferro_2,
    h=h_ferro_2,
    n_runs=50,
    n_iter=n_iter,
    dt=dt,
    seed=42,
)

res_ferro_2

,instance,N,variant,E_exact,E_best,mean_energy,mean_gap,success_rate,mean_runtime_sec,n_exact_states
0,2-spin ferromagnetic,2,BSB,-1.0,-1.0,-1.0,0.0,1.0,0.028350,2
1,2-spin ferromagnetic,2,DSB,-1.0,-1.0,-1.0,0.0,1.0,0.028892,2


# Тест 2: две антиферромагнитные спины

In [35]:
J_antiferro_2 = np.array([
    [0.0, -1.0],
    [-1.0, 0.0],
])

h_antiferro_2 = np.zeros(2)

E_exact, states_exact = brute_force_ising(J_antiferro_2, h_antiferro_2)

E_exact, states_exact

(-1.0, [array([-1,  1]), array([ 1, -1])])

In [36]:
res_antiferro_2 = evaluate_instance(
    name="2-spin antiferromagnetic",
    J=J_antiferro_2,
    h=h_antiferro_2,
    n_runs=50,
    n_iter=n_iter,
    dt=dt,
    seed=42,
)

res_antiferro_2

,instance,N,variant,E_exact,E_best,mean_energy,mean_gap,success_rate,mean_runtime_sec,n_exact_states
0,2-spin antiferromagnetic,2,BSB,-1.0,-1.0,-1.0,0.0,1.0,0.028082,2
1,2-spin antiferromagnetic,2,DSB,-1.0,-1.0,-1.0,0.0,1.0,0.028849,2


In [37]:
summary_basic_2spin = pd.concat(
    [res_ferro_2, res_antiferro_2],
    ignore_index=True
)

summary_basic_2spin

,instance,N,variant,E_exact,E_best,mean_energy,mean_gap,success_rate,mean_runtime_sec,n_exact_states
0,2-spin ferromagnetic,2,BSB,-1.0,-1.0,-1.0,0.0,1.0,0.028350,2
1,2-spin ferromagnetic,2,DSB,-1.0,-1.0,-1.0,0.0,1.0,0.028892,2
2,2-spin antiferromagnetic,2,BSB,-1.0,-1.0,-1.0,0.0,1.0,0.028082,2
3,2-spin antiferromagnetic,2,DSB,-1.0,-1.0,-1.0,0.0,1.0,0.028849,2


# Дополнительные простые Ising-модели

После проверки двухспиновых моделей рассмотрим три малые задачи:

1. ферромагнитная цепочка;
2. антиферромагнитная цепочка;
3. фрустрированный треугольник.

Первые две задачи имеют простую аналитическую структуру оптимума. Фрустрированный треугольник важен как первый пример, где нельзя одновременно удовлетворить все парные взаимодействия.

Берём N=6.

In [38]:
def build_chain_J(N, coupling=1.0):
    """
    Строит Ising-цепочку из N спинов:

        J_{i,i+1} = coupling

    coupling = +1  -> ферромагнитная цепочка
    coupling = -1  -> антиферромагнитная цепочка
    """
    J = np.zeros((N, N), dtype=float)

    for i in range(N - 1):
        J[i, i + 1] = coupling
        J[i + 1, i] = coupling

    return J

### Ферромагнитная цепочка

Берём N=6. Для ферромагнитной цепочки минимум достигается, когда все спины одинаковые.

In [39]:
N_chain = 6

J_ferro_chain = build_chain_J(N_chain, coupling=1.0)
h_ferro_chain = np.zeros(N_chain)

E_exact, states_exact = brute_force_ising(J_ferro_chain, h_ferro_chain)

E_exact, states_exact[:3], len(states_exact)

(-5.0, [array([-1, -1, -1, -1, -1, -1]), array([1, 1, 1, 1, 1, 1])], 2)

In [40]:
TEST_DT = 0.05
TEST_N_ITER = 1000
TEST_N_RUNS = 50
TEST_SEED = 42

In [41]:
res_ferro_chain = evaluate_instance(
    name="ferromagnetic chain",
    J=J_ferro_chain,
    h=h_ferro_chain,
    n_runs=TEST_N_RUNS,
    n_iter=TEST_N_ITER,
    dt=TEST_DT,
    seed=TEST_SEED,
)

res_ferro_chain

,instance,N,variant,E_exact,E_best,mean_energy,mean_gap,success_rate,mean_runtime_sec,n_exact_states
0,ferromagnetic chain,6,BSB,-5.0,-5.0,-5.0,0.0,1.0,0.028231,2
1,ferromagnetic chain,6,DSB,-5.0,-5.0,-5.0,0.0,1.0,0.029479,2


### Антиферромагнитная цепочка

Для антиферромагнитной цепочки минимум достигается при чередовании знаков:

In [42]:
J_antiferro_chain = build_chain_J(N_chain, coupling=-1.0)
h_antiferro_chain = np.zeros(N_chain)

E_exact, states_exact = brute_force_ising(J_antiferro_chain, h_antiferro_chain)

E_exact, states_exact[:3], len(states_exact)

(-5.0, [array([-1,  1, -1,  1, -1,  1]), array([ 1, -1,  1, -1,  1, -1])], 2)

In [43]:
res_antiferro_chain = evaluate_instance(
    name="antiferromagnetic chain",
    J=J_antiferro_chain,
    h=h_antiferro_chain,
    n_runs=TEST_N_RUNS,
    n_iter=TEST_N_ITER,
    dt=TEST_DT,
    seed=TEST_SEED,
)

res_antiferro_chain

,instance,N,variant,E_exact,E_best,mean_energy,mean_gap,success_rate,mean_runtime_sec,n_exact_states
0,antiferromagnetic chain,6,BSB,-5.0,-5.0,-5.0,0.0,1.0,0.031095,2
1,antiferromagnetic chain,6,DSB,-5.0,-5.0,-5.0,0.0,1.0,0.030553,2


## Фрустрированный треугольник

Теперь важный тест. Берём три спина и все три связи делаем антиферромагнитными:
$$
J_{12} = J_{23} = J_{13} = -1
$$


В треугольнике невозможно сделать так, чтобы все три пары спинов были противоположными. Поэтому оптимум не равен -3, а равен -1.

In [44]:
J_frustrated_triangle = np.array([
    [0.0, -1.0, -1.0],
    [-1.0, 0.0, -1.0],
    [-1.0, -1.0, 0.0],
])

h_frustrated_triangle = np.zeros(3)

E_exact, states_exact = brute_force_ising(
    J_frustrated_triangle,
    h_frustrated_triangle
)

E_exact, states_exact, len(states_exact)

(-1.0,
 [array([-1, -1,  1]),
  array([-1,  1, -1]),
  array([-1,  1,  1]),
  array([ 1, -1, -1]),
  array([ 1, -1,  1]),
  array([ 1,  1, -1])],
 6)

In [45]:
res_frustrated_triangle = evaluate_instance(
    name="frustrated triangle",
    J=J_frustrated_triangle,
    h=h_frustrated_triangle,
    n_runs=TEST_N_RUNS,
    n_iter=TEST_N_ITER,
    dt=TEST_DT,
    seed=TEST_SEED,
)

res_frustrated_triangle

,instance,N,variant,E_exact,E_best,mean_energy,mean_gap,success_rate,mean_runtime_sec,n_exact_states
0,frustrated triangle,3,BSB,-1.0,-1.0,-1.0,0.0,1.0,0.028198,6
1,frustrated triangle,3,DSB,-1.0,-1.0,-1.0,0.0,1.0,0.029745,6


# Итоговая таблица по простым моделям

In [46]:
tables = []

if "summary_basic_2spin_stable" in globals():
    tables.append(summary_basic_2spin_stable)
elif "summary_basic_2spin" in globals():
    tables.append(summary_basic_2spin)

tables.extend([
    res_ferro_chain,
    res_antiferro_chain,
    res_frustrated_triangle,
])

summary_simple_ising = pd.concat(tables, ignore_index=True)

summary_simple_ising

,instance,N,variant,E_exact,E_best,mean_energy,mean_gap,success_rate,mean_runtime_sec,n_exact_states
0,2-spin ferromagnetic,2,BSB,-1.0,-1.0,-1.0,0.0,1.0,0.028350,2
1,2-spin ferromagnetic,2,DSB,-1.0,-1.0,-1.0,0.0,1.0,0.028892,2
2,2-spin antiferromagnetic,2,BSB,-1.0,-1.0,-1.0,0.0,1.0,0.028082,2
3,2-spin antiferromagnetic,2,DSB,-1.0,-1.0,-1.0,0.0,1.0,0.028849,2
4,ferromagnetic chain,6,BSB,-5.0,-5.0,-5.0,0.0,1.0,0.028231,2
5,ferromagnetic chain,6,DSB,-5.0,-5.0,-5.0,0.0,1.0,0.029479,2
6,antiferromagnetic chain,6,BSB,-5.0,-5.0,-5.0,0.0,1.0,0.031095,2
7,antiferromagnetic chain,6,DSB,-5.0,-5.0,-5.0,0.0,1.0,0.030553,2
8,frustrated triangle,3,BSB,-1.0,-1.0,-1.0,0.0,1.0,0.028198,6
9,frustrated triangle,3,DSB,-1.0,-1.0,-1.0,0.0,1.0,0.029745,6


In [47]:
summary_simple_ising_compact = summary_simple_ising[
    [
        "instance",
        "N",
        "variant",
        "E_exact",
        "E_best",
        "mean_gap",
        "success_rate",
        "mean_runtime_sec",
        "n_exact_states",
    ]
].copy()

summary_simple_ising_compact["mean_runtime_ms"] = (
    1000 * summary_simple_ising_compact["mean_runtime_sec"]
)

summary_simple_ising_compact = summary_simple_ising_compact.drop(
    columns=["mean_runtime_sec"]
)

summary_simple_ising_compact

,instance,N,variant,E_exact,E_best,mean_gap,success_rate,n_exact_states,mean_runtime_ms
0,2-spin ferromagnetic,2,BSB,-1.0,-1.0,0.0,1.0,2,28.349920
1,2-spin ferromagnetic,2,DSB,-1.0,-1.0,0.0,1.0,2,28.891606
2,2-spin antiferromagnetic,2,BSB,-1.0,-1.0,0.0,1.0,2,28.081650
3,2-spin antiferromagnetic,2,DSB,-1.0,-1.0,0.0,1.0,2,28.848782
4,ferromagnetic chain,6,BSB,-5.0,-5.0,0.0,1.0,2,28.231302
5,ferromagnetic chain,6,DSB,-5.0,-5.0,0.0,1.0,2,29.478624
6,antiferromagnetic chain,6,BSB,-5.0,-5.0,0.0,1.0,2,31.094532
7,antiferromagnetic chain,6,DSB,-5.0,-5.0,0.0,1.0,2,30.552618
8,frustrated triangle,3,BSB,-1.0,-1.0,0.0,1.0,6,28.197732
9,frustrated triangle,3,DSB,-1.0,-1.0,0.0,1.0,6,29.744776


# Проверка Ising-модели с локальным полем

До этого рассматривались модели без локального поля, то есть $h_i = 0$.
Теперь проверим, что реализация корректно учитывает линейный член:

$$
E(s) = - \sum_{i<j} J_{ij}s_i s_j - \sum_i h_i s_i.
$$

Если $h_i > 0$, энергия уменьшается при $s_i = +1$.
Если $h_i < 0$, энергия уменьшается при $s_i = -1$.

## Простая модель только с локальным полем

Здесь специально берём J=0, чтобы проверить именно поле.

In [48]:
J_field_only = np.zeros((2, 2), dtype=float)

h_field_only = np.array([1.0, -0.7])

E_exact, states_exact = brute_force_ising(J_field_only, h_field_only)

E_exact, states_exact, len(states_exact)

(-1.7, [array([ 1, -1])], 1)

In [49]:
res_field_only = evaluate_instance(
    name="field-only model",
    J=J_field_only,
    h=h_field_only,
    n_runs=TEST_N_RUNS,
    n_iter=TEST_N_ITER,
    dt=TEST_DT,
    seed=TEST_SEED,
)

res_field_only

,instance,N,variant,E_exact,E_best,mean_energy,mean_gap,success_rate,mean_runtime_sec,n_exact_states
0,field-only model,2,BSB,-1.7,-1.7,-1.7,0.0,1.0,0.029952,1
1,field-only model,2,DSB,-1.7,-1.7,-1.7,0.0,1.0,0.030490,1


## Модель со связями и локальным полем

In [50]:
J_mixed_field = np.array([
    [0.0, 1.0, -0.5],
    [1.0, 0.0, 0.8],
    [-0.5, 0.8, 0.0],
])

h_mixed_field = np.array([0.4, -0.2, 0.7])

E_exact, states_exact = brute_force_ising(J_mixed_field, h_mixed_field)

E_exact, states_exact, len(states_exact)

(-2.2, [array([1, 1, 1])], 1)

In [51]:
res_mixed_field = evaluate_instance(
    name="mixed J and field model",
    J=J_mixed_field,
    h=h_mixed_field,
    n_runs=TEST_N_RUNS,
    n_iter=TEST_N_ITER,
    dt=TEST_DT,
    seed=TEST_SEED,
)

res_mixed_field

,instance,N,variant,E_exact,E_best,mean_energy,mean_gap,success_rate,mean_runtime_sec,n_exact_states
0,mixed J and field model,3,BSB,-2.2,-2.2,-2.20,0.00,1.00,0.028318,1
1,mixed J and field model,3,DSB,-2.2,-2.2,-2.06,0.14,0.86,0.028761,1


# Супер итог

In [52]:
summary_431 = pd.concat(
    [
        summary_simple_ising,
        res_field_only,
        res_mixed_field,
    ],
    ignore_index=True,
)

summary_431_compact = summary_431[
    [
        "instance",
        "N",
        "variant",
        "E_exact",
        "E_best",
        "mean_gap",
        "success_rate",
        "mean_runtime_sec",
        "n_exact_states",
    ]
].copy()

summary_431_compact["mean_runtime_ms"] = (
    1000 * summary_431_compact["mean_runtime_sec"]
)

summary_431_compact = summary_431_compact.drop(
    columns=["mean_runtime_sec"]
)

summary_431_compact

,instance,N,variant,E_exact,E_best,mean_gap,success_rate,n_exact_states,mean_runtime_ms
0,2-spin ferromagnetic,2,BSB,-1.0,-1.0,0.00,1.00,2,28.349920
1,2-spin ferromagnetic,2,DSB,-1.0,-1.0,0.00,1.00,2,28.891606
2,2-spin antiferromagnetic,2,BSB,-1.0,-1.0,0.00,1.00,2,28.081650
3,2-spin antiferromagnetic,2,DSB,-1.0,-1.0,0.00,1.00,2,28.848782
4,ferromagnetic chain,6,BSB,-5.0,-5.0,0.00,1.00,2,28.231302
5,ferromagnetic chain,6,DSB,-5.0,-5.0,0.00,1.00,2,29.478624
6,antiferromagnetic chain,6,BSB,-5.0,-5.0,0.00,1.00,2,31.094532
7,antiferromagnetic chain,6,DSB,-5.0,-5.0,0.00,1.00,2,30.552618
8,frustrated triangle,3,BSB,-1.0,-1.0,0.00,1.00,6,28.197732
9,frustrated triangle,3,DSB,-1.0,-1.0,0.00,1.00,6,29.744776


# Сохранение результатов

In [53]:
summary_431_compact.to_csv(
    "summary_04_03_01_small_ising_known_optimum.csv",
    index=False
)

print("Saved: summary_04_03_01_small_ising_known_optimum.csv")

Saved: summary_04_03_01_small_ising_known_optimum.csv
